# AI-Based Spam & Threat Detection System
### BCA (AI & ML) Project - Google Colab Notebook
---
This notebook contains the complete end-to-end implementation of the **AI-Based Spam Detection System** as specified in the SRS & PRD:
1. **Dataset Download & Ingestion:** UCI SMS Spam Collection dataset.
2. **Exploratory Data Analysis (EDA):** Class distribution and text characteristics.
3. **Preprocessing & Feature Extraction:** TF-IDF n-gram vectorization.
4. **Model Training:** Logistic Regression with probability estimates.
5. **Rigorous Evaluation:** Confusion Matrix, Accuracy, Precision, Recall, and F1-score.
6. **Model Artifact Export:** Serialization of `model.pkl`, `vectorizer.pkl`, and `metrics.json`.
7. **Module 2 - Link / Phishing URL Detection Engine:** Structural URL heuristics.
8. **Module 3 - File & Batch Document Detection:** Bulk message processing.
9. **Interactive Testing:** Instant classification widgets and test functions.

## 1. Setup & Dependencies
Install and import all necessary scientific and ML packages.

In [ ]:
# Install required packages (standard in Colab, but ensures latest compatibility)
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib urllib3

import os
import re
import json
import joblib
import urllib.request
from datetime import datetime
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

print("[OK] All libraries imported successfully!")

## 2. Dataset Download & Loading
We automatically download the verified **UCI SMS Spam Collection** dataset (5,574 labeled messages).

In [ ]:
# Download UCI SMS Spam Collection dataset
DATASET_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
DATASET_FILE = "SMSSpamCollection.tsv"

if not os.path.exists(DATASET_FILE):
    print(f"Downloading dataset from {DATASET_URL}...")
    urllib.request.urlretrieve(DATASET_URL, DATASET_FILE)
    print("Download complete!")
else:
    print("Dataset already present locally.")

# Load dataset into pandas
df = pd.read_csv(DATASET_FILE, sep='\t', names=['label', 'message'], encoding='utf-8')

# Clean and preprocess columns
df = df.dropna(subset=['label', 'message'])
df['label'] = df['label'].astype(str).str.strip().str.lower()
df['message'] = df['message'].astype(str).str.strip()
df = df[df['label'].isin(['ham', 'spam'])]

# Numerical target: 0 for Ham, 1 for Spam
df['target'] = df['label'].map({'ham': 0, 'spam': 1})
df['char_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

print(f"Total valid messages: {len(df)}")
print(f"Ham count: {(df['label'] == 'ham').sum()} | Spam count: {(df['label'] == 'spam').sum()}")
df.head()

## 3. Exploratory Data Analysis (EDA)
Let's visualize the class distribution and inspect how spam messages differ in length compared to genuine (ham) messages.

In [ ]:
plt.figure(figsize=(14, 5))

# 1. Class Distribution Pie / Bar Chart
plt.subplot(1, 2, 1)
counts = df['label'].value_counts()
colors = ['#2ecc71', '#ff4d6d']
plt.bar(['Ham (Genuine)', 'Spam'], counts.values, color=colors, edgecolor='black', width=0.5)
plt.title("Class Distribution (Ham vs. Spam)", fontsize=13, fontweight='bold')
plt.ylabel("Number of Messages")
for i, v in enumerate(counts.values):
    plt.text(i, v + 50, f"{v} ({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')

# 2. Message Length Histogram
plt.subplot(1, 2, 2)
sns.histplot(df[df['label']=='ham']['char_length'], color='#2ecc71', label='Ham', kde=True, bins=40, stat='density')
sns.histplot(df[df['label']=='spam']['char_length'], color='#ff4d6d', label='Spam', kde=True, bins=40, stat='density')
plt.xlim(0, 300)
plt.title("Message Length Distribution by Class", fontsize=13, fontweight='bold')
plt.xlabel("Character Length")
plt.ylabel("Density")
plt.legend()

plt.tight_layout()
plt.show()

## 4. Train / Test Split (Stratified 80/20)
To prevent data leakage and guarantee that spam class proportions are maintained in test evaluation, we use a stratified 80/20 split with a fixed random seed (`random_state=42`).

In [ ]:
X = df['message']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size:  {len(X_test)} samples")
print(f"Spam in train: {y_train.sum()} ({y_train.sum()/len(y_train)*100:.2f}%)")
print(f"Spam in test:  {y_test.sum()} ({y_test.sum()/len(y_test)*100:.2f}%)")

## 5. TF-IDF Feature Extraction & Logistic Regression Model
We fit the TF-IDF vectorizer **strictly on training data** to avoid data snooping/leakage, then train a Logistic Regression classifier.

In [ ]:
# 1. Fit TF-IDF Vectorizer on train set only
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 2. Train Logistic Regression Classifier
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

print("[OK] Model training complete!")

## 6. Comprehensive Model Evaluation
Evaluating test set predictions across **Accuracy**, **Precision (Spam)**, **Recall (Spam)**, and **F1-Score (Spam)**, followed by Confusion Matrix and ROC curve visualizations.

In [ ]:
# Predictions on unseen test set
y_pred = model.predict(X_test_vec)
y_prob = model.predict_proba(X_test_vec)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label=1)
rec = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)

print("=" * 50)
print("       MODEL EVALUATION SCORES (TEST SET)")
print("=" * 50)
print(f"  Accuracy:         {acc * 100:.2f}%")
print(f"  Precision (Spam): {prec * 100:.2f}%")
print(f"  Recall (Spam):    {rec * 100:.2f}%")
print(f"  F1-Score (Spam):  {f1 * 100:.2f}%")
print("=" * 50)

print("\nDetailed Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Ham (Genuine)', 'Spam']))

# Visualization: Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Predicted Ham', 'Predicted Spam'],
            yticklabels=['Actual Ham', 'Actual Spam'])
axes[0].set_title("Confusion Matrix (Test Set)", fontsize=13, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#2a52be', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontsize=13, fontweight='bold')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 7. Model Artifact Export
Save `model.pkl`, `vectorizer.pkl`, and `metrics.json` so they can be loaded by the web backend or downloaded from Colab.

In [ ]:
# Save model, vectorizer, and metrics
joblib.dump(model, 'model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

metrics_dict = {
    "accuracy": round(acc * 100, 2),
    "precision": round(prec * 100, 2),
    "recall": round(rec * 100, 2),
    "f1_score": round(f1 * 100, 2),
    "total_samples": int(len(df)),
    "train_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open('metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=4)

print("[OK] Saved model.pkl, vectorizer.pkl, and metrics.json successfully!")

## 8. Module 2: Link / Phishing URL Detection Engine
Analyzes links using **11+ structural security heuristics** (suspicious TLDs, URL shorteners, direct IP hosting, deceptive keyword lures, typosquatting, etc.).

In [ ]:
SUSPICIOUS_TLDS = {'xyz', 'top', 'club', 'tk', 'ml', 'ga', 'cf', 'gq', 'work', 'click', 'loan', 'fit', 'racing', 'download'}
URL_SHORTENERS = {'bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly', 'is.gd', 'buff.ly', 'cutt.ly', 'rebrand.ly'}
PHISHING_KEYWORDS = ['login', 'signin', 'verify', 'account', 'update', 'banking', 'secure', 'wallet', 'password', 'claim', 'prize', 'reward', 'urgent', 'suspend']
TYPOSQUATTING_PATTERNS = [r'paypa[l1i]', r'g[o0]{2}gle', r'micr[o0]s[o0]ft', r'amaz[o0]n', r'netfl[i1]x']

def analyze_url(url_string):
    """Analyzes a URL against 11+ phishing and security heuristics."""
    url = (url_string or "").strip()
    if not url:
        return {"error": "URL cannot be empty"}
    
    url_to_parse = url if re.match(r'^[a-zA-Z]+://', url) else 'http://' + url
    parsed = urlparse(url_to_parse)
    netloc = parsed.netloc.lower()
    full_url = url.lower()
    
    score = 0
    indicators = []
    
    if parsed.scheme == 'http':
        score += 15
        indicators.append("Non-secure HTTP protocol used.")
    if re.match(r'^(\d{1,3}\.){3}\d{1,3}(:\d+)?$', netloc):
        score += 40
        indicators.append("Direct IP address used as hostname.")
    
    host_clean = netloc.split(':')[0]
    if host_clean in URL_SHORTENERS:
        score += 25
        indicators.append(f"Masked URL shortener '{host_clean}' detected.")
    
    tld = host_clean.split('.')[-1] if '.' in host_clean else ''
    if tld in SUSPICIOUS_TLDS:
        score += 30
        indicators.append(f"High-risk TLD '.{tld}' detected.")
        
    if host_clean.count('.') > 3:
        score += 20
        indicators.append("Excessive subdomain depth detected.")
    if '@' in url_to_parse:
        score += 35
        indicators.append("Contains '@' symbol (URL obfuscation trick).")
    if len(url) > 75:
        score += 15
        indicators.append(f"Abnormally long URL length ({len(url)} chars).")
        
    matched_kw = [kw for kw in PHISHING_KEYWORDS if kw in full_url]
    if matched_kw:
        score += min(30, len(matched_kw) * 15)
        indicators.append(f"Deceptive lure keywords found: {', '.join(matched_kw[:3])}")
        
    for pattern in TYPOSQUATTING_PATTERNS:
        if re.search(pattern, host_clean) and not re.search(r'^(www\.)?(google\.com|paypal\.com|microsoft\.com|amazon\.com|netflix\.com)$', host_clean):
            score += 35
            indicators.append("Suspected brand impersonation / typosquatting.")
            break
            
    risk_score = min(99, max(5, score))
    risk_level = "High Risk Phishing" if risk_score >= 55 else ("Suspicious Link" if risk_score >= 30 else "Safe / Legitimate Link")
    
    return {
        "url": url,
        "hostname": host_clean,
        "risk_level": risk_level,
        "risk_score": risk_score,
        "is_malicious": risk_score >= 30,
        "indicators": indicators or ["No suspicious anomalies detected"]
    }

print("[OK] URL analysis engine loaded!")

## 9. Module 3: File & Batch Message Detection Engine
Allows batch processing of multiple text messages from a list, CSV file, or document.

In [ ]:
def process_batch_messages(messages_list):
    """Classifies a list of messages in bulk using the trained model."""
    valid_msgs = [str(m).strip() for m in messages_list if len(str(m).strip()) >= 3]
    if not valid_msgs:
        return {"error": "No valid messages provided."}
    
    vecs = vectorizer.transform([m.lower() for m in valid_msgs])
    preds = model.predict(vecs)
    probs = model.predict_proba(vecs)
    
    results = []
    spam_count = 0
    for i, msg in enumerate(valid_msgs):
        p = preds[i]
        is_spam = bool(p == 1)
        conf = round(float(probs[i][p]) * 100, 2)
        if is_spam:
            spam_count += 1
        results.append({
            "index": i + 1,
            "message": msg,
            "verdict": "Spam" if is_spam else "Not Spam",
            "confidence": conf
        })
        
    total = len(valid_msgs)
    spam_ratio = round((spam_count / total * 100), 1)
    
    return {
        "total_scanned": total,
        "spam_count": spam_count,
        "ham_count": total - spam_count,
        "spam_ratio": spam_ratio,
        "threat_level": "High Threat" if spam_ratio >= 40 else ("Moderate Threat" if spam_ratio > 0 else "Clean Document"),
        "results": results
    }

print("[OK] Batch processing engine loaded!")

## 10. Interactive Testing (Single Message, URL, and Batch File)
Run the cells below to test the engines with your own inputs or standard SRS test cases.

In [ ]:
# Helper function to test single message
def predict_message(message_text):
    clean = (message_text or "").strip()
    if not clean:
        return {"error": "Message cannot be empty"}
    if len(clean) > 1000:
        return {"error": "Message exceeds 1000 character limit"}
        
    vec = vectorizer.transform([clean.lower()])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    
    label = "Spam" if pred == 1 else "Not Spam"
    confidence = round(float(prob[pred]) * 100, 2)
    
    print(f"\n{'='*55}")
    print(f"Input Text:   '{clean}'")
    print(f"Verdict:      {label.upper()}")
    print(f"Confidence:   {confidence}%")
    print(f"{'='*55}")
    return {"label": label, "confidence": confidence}

# 1. Test SRS Test Case TC-1 (Spam)
predict_message("Congratulations! You won a free prize. Call now to claim.")

# 2. Test SRS Test Case TC-2 (Ham)
predict_message("Hey, are we meeting at 5 pm today?")

# 3. Test Phishing URL
print("\n--- URL Analysis Sample ---")
url_res = analyze_url("http://account-verify-login-paypal.xyz/auth?claim=1")
print(f"URL:      {url_res['url']}")
print(f"Verdict:  {url_res['risk_level']} ({url_res['risk_score']}% threat risk)")
print(f"Signals:  {url_res['indicators']}")

# 4. Test Batch File Scanner Sample
print("\n--- Batch Messages Sample ---")
sample_batch = [
    "URGENT! You have won a £1,000 cash prize. Call 09061701461 to claim now!",
    "Can you please send me the notes from today's lecture?",
    "Free entry in 2 a weekly competition to win tickets! Text WIN to 87121",
    "See you at home for dinner tonight."
]
batch_res = process_batch_messages(sample_batch)
print(f"Total: {batch_res['total_scanned']} | Spam: {batch_res['spam_count']} | Ham: {batch_res['ham_count']} | Level: {batch_res['threat_level']}")
for r in batch_res['results']:
    print(f"  [{r['verdict']}] ({r['confidence']}%) - {r['message'][:50]}...")

## 11. Interactive Colab UI Widget
This cell renders interactive textboxes and buttons directly inside Google Colab so you can type and inspect any message or URL with one click!

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

output_area = widgets.Output()

text_area = widgets.Textarea(
    value="Congratulations! You won a free prize. Call now to claim.",
    placeholder='Type or paste SMS message or URL here...',
    description='Input:',
    layout=widgets.Layout(width='85%', height='100px')
)

btn_msg = widgets.Button(description="Check Message", button_style='danger', icon='shield')
btn_url = widgets.Button(description="Check Link / URL", button_style='warning', icon='link')

def on_msg_clicked(b):
    with output_area:
        output_area.clear_output()
        res = predict_message(text_area.value)
        color = "#ff4d6d" if res.get('label') == 'Spam' else "#2ecc71"
        display(HTML(f"<div style='background:#111b3a; color:#fff; padding:15px; border-radius:8px; border-left:5px solid {color}; margin-top:10px;'>"\
                     f"<h4 style='color:{color}; margin:0;'>Verdict: {res.get('label')}</h4>"\
                     f"<p style='margin:5px 0 0 0;'>Model Confidence: <b>{res.get('confidence')}%</b></p></div>"))

def on_url_clicked(b):
    with output_area:
        output_area.clear_output()
        res = analyze_url(text_area.value)
        color = "#ff4d6d" if res.get('is_malicious') else "#2ecc71"
        display(HTML(f"<div style='background:#111b3a; color:#fff; padding:15px; border-radius:8px; border-left:5px solid {color}; margin-top:10px;'>"\
                     f"<h4 style='color:{color}; margin:0;'>Verdict: {res.get('risk_level')} ({res.get('risk_score')}% risk)</h4>"\
                     f"<p style='margin:5px 0 0 0;'>Findings: {', '.join(res.get('indicators', []))}</p></div>"))

btn_msg.on_click(on_msg_clicked)
btn_url.on_click(on_url_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>🛡️ SpamGuard.AI - Interactive Colab Inspector</h3>"),
    text_area,
    widgets.HBox([btn_msg, btn_url]),
    output_area
]))